# Phase 4: Feature Engineering

In this phase, we generate features to help our predictive models learn temporal dynamics, patterns, and interactions. We'll create time-based features, lag features, rolling statistics, and interaction terms. Finally, we'll create the target variables and split the dataset.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset
We load the cleaned dataset. Note: Our dataset `clean_air_quality.csv` is at a **daily** frequency. Thus, a lag of 24 represents 24 days.

In [ ]:
df = pd.read_csv('../data/processed/clean_air_quality.csv', parse_dates=['date'])
df = df.sort_values(by=['city', 'date']).reset_index(drop=True)
print(f"Initial shape: {df.shape}")

## 2. Time Features
Extracting Year, Month, Day, Weekday, Quarter, and Season.

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]: return 1 # Winter
    elif month in [3, 4, 5]: return 2 # Summer
    elif month in [6, 7, 8, 9]: return 3 # Monsoon
    else: return 4 # Post-Monsoon

df['season'] = df['month'].apply(get_season)
print("Time features created.")

## 3. Lag Features
Creating historical lags for AQI. Since this is grouped by city, we use `groupby().shift()`.

In [ ]:
lags = [1, 3, 6, 12, 24]
for lag in lags:
    df[f'aqi_lag_{lag}'] = df.groupby('city')['aqi'].shift(lag)
print("Lag features created.")

## 4. Rolling Features
Calculating rolling mean, standard deviation, max, and min over a 7-day window.

In [ ]:
window = 7
df['rolling_mean'] = df.groupby('city')['aqi'].transform(lambda x: x.rolling(window, min_periods=1).mean())
df['rolling_std'] = df.groupby('city')['aqi'].transform(lambda x: x.rolling(window, min_periods=1).std())
df['rolling_max'] = df.groupby('city')['aqi'].transform(lambda x: x.rolling(window, min_periods=1).max())
df['rolling_min'] = df.groupby('city')['aqi'].transform(lambda x: x.rolling(window, min_periods=1).min())
print("Rolling features created.")

## 5. Interaction Features
Creating cross-pollutant interactions.

In [ ]:
if 'pm2_5' in df.columns and 'pm10' in df.columns:
    df['pm2_5_x_pm10'] = df['pm2_5'] * df['pm10']
if 'pm2_5' in df.columns and 'co' in df.columns:
    df['pm2_5_x_co'] = df['pm2_5'] * df['co']
print("Interaction features created.")

## 6. Prediction Targets
We create target columns: `aqi_24`, `aqi_48`, and `aqi_72`. Since the dataset is daily, we adapt this to mean 1 day (24h), 2 days (48h), and 3 days (72h) ahead forecasting targets.

In [ ]:
df['aqi_24'] = df.groupby('city')['aqi'].shift(-1)
df['aqi_48'] = df.groupby('city')['aqi'].shift(-2)
df['aqi_72'] = df.groupby('city')['aqi'].shift(-3)

# Drop rows where targets are NaN (due to shifting at the end of the time series)
df = df.dropna(subset=['aqi_24', 'aqi_48', 'aqi_72']).reset_index(drop=True)
print("Prediction targets created.")

## 7. Train / Validation / Test Split
Since this is time-series data, we split chronologically. 
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%

In [ ]:
def time_series_split(group):
    n = len(group)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    
    group['split'] = 'test'
    group.iloc[:train_end, group.columns.get_loc('split')] = 'train'
    group.iloc[train_end:val_end, group.columns.get_loc('split')] = 'val'
    return group

df = df.groupby('city', group_keys=False).apply(time_series_split)
print("Data split into Train/Validation/Test chronologically per city.")
print(df['split'].value_counts(normalize=True))

## 8. Save Processed Dataset

In [ ]:
OUTPUT_PATH = '../data/processed/featured_air_quality.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"Feature engineered dataset saved to {OUTPUT_PATH}")